## INTERROGATION AGENT

FILE STRUCTURE

- Preparation
    - imports
    - loads & initializations
- Game System
    - GameState
    - GameController
- Agentic System
    - Tools
    - Pipeline
    - Memory
- Game Loop / Start interrogation

### Preparation

imports

In [1]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.tools import tool
from langchain_core.tools import tool
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from pinecone import Pinecone
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

from pydantic import BaseModel

loads & initializations

In [2]:
# load API keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

Initializations

In [3]:
# connect to pinecone
pc = Pinecone(api_key=PINECONE_API_KEY) 

In [4]:
# load embeddings model 
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [5]:
# load vector stores
detective_vector_store = PineconeVectorStore(
    index_name="detective-questions",
    embedding=embeddings
)

suspect_vector_store = PineconeVectorStore(
    index_name="suspect-behaviour",
    embedding=embeddings
)

In [6]:
# initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

Case facts

In [7]:
# hardcoded from wikipedia article

CASE_FACTS = """
On February 24, 1986, the body of Sherri Rasmussen (born February 7, 1957[1]) was found in the condominium she shared with her husband, John Ruetten, in the Van Nuys neighborhood of Los Angeles, California, United States. 

She had been beaten and shot three times.

The Los Angeles Police Department (LAPD) initially considered the case a botched burglary and were unable to identify a suspect. Rasmussen's father believed that LAPD officer Stephanie Lazarus, formerly in a relationship with Ruetten, was a prime suspect due to her continued attraction to Ruetten and confrontations with Rasmussen. 

The investigation stalled after several months; the case went cold for over two decades afterwards.

While an undergraduate at the University of California, Los Angeles (UCLA) from 1978 to 1982, John Ruetten, a mechanical engineering major from San Diego, occasionally dated Stephanie Lazarus, a fellow Dykstra Hall resident from Simi Valley majoring in political science and sociology.[2] Their friends said she seemed to take the relationship more seriously than he did.[3] Both were avid athletes; Lazarus played on UCLA's junior varsity women's basketball team. She stole Ruetten's clothes when he showered and photographed him in his underwear while he slept. Ruetten never considered the relationship anything more than "necking and fooling around". They had sex for the first time after he graduated.[4] After that they saw each other two or three times a month, occasionally taking trips together. Some of those encounters resulted in sex.[5]

Following his 1981 graduation Ruetten accepted a job with Dataproducts, a maker of computer peripherals.[6] After graduating, a year later, Lazarus considered law school.[7] She decided on law enforcement instead and was admitted to the city's police academy. At that time, the Los Angeles Police Department (LAPD) was trying to increase the number of women on the force in response to a consent decree following a sex-discrimination lawsuit brought by former female officers. After completing a special eight-week pre-academy training program,[8] she graduated and became a uniformed officer in 1983.[9] Classmates recalled that she was particularly tenacious during physical combat training, especially in exercises where trainees had to retain control of a weapon. During her training, Lazarus described Ruetten as her college boyfriend,[10] and she was his date at Dataproducts' 1983 Christmas party, where they had a photo taken together.[11] He later testified that they had sex "20 to 30 times" between 1981 and 1984, but that he never considered her his girlfriend.[12] Upon graduation Lazarus settled in a Granada Hills condo and purchased a Smith & Wesson Model 49 .38-caliber revolver through the department as her backup firearm.[13]

Ruetten later met Sherri Rasmussen, a graduate of Loma Linda University who was on a fast career track in critical care nursing. Born in Walla Walla, Washington and raised in Arizona, Rasmussen began college at La Sierra University at 16 after graduating from Thunderbird Adventist Academy. After her freshman year she was accepted at Loma Linda's nursing program and transferred there.[14] Upon graduating, she worked at UCLA Medical Center's coronary care unit and studied for the master's in nursing UCLA awarded her in 1980.[15] Afterwards, her father, Nels Rasmussen, bought Sherri a condo in Van Nuys with a drive-in garage so she would not have to walk along the street after returning from work late at night. She paid him rent equal to his monthly mortgage payment.[16]

After earning her degree, Rasmussen was promoted to head nurse of UCLA's coronary care and observation unit.[16] She also was appointed an unpaid assistant clinical professor of nursing, giving lectures to students.[17] By her late 20s Rasmussen was the director of nursing at Glendale Adventist Medical Center. She also gave presentations and taught classes for fellow nurses.[18] At a June 1984 party Sherri met Ruetten; they began dating shortly afterwards.[19]

For her first assignment as a uniformed LAPD officer, Lazarus drew the Hollywood division, an area notorious at the time for high levels of street crime. Morale among the predominantly male officers was poor in the wake of the "Hollywood Burglars" scandal in which 14 officers were ultimately fired after two were prosecuted for burglarizing video stores while on duty. Chief Daryl Gates closely oversaw the internal affairs investigation. He observed later that Hollywood seemed to have higher rates of police misconduct than other divisions. "There is something about the place," he wrote, "an almost carnival atmosphere that suggests 'here, anything goes'".[20]

A female colleague of Lazarus's at Hollywood, one of the few other women in the division and a teammate on the department's women's basketball team, recalls her as being an asset to the team primarily through her tight and physical defensive play and willingness to work with others.[20][a] Teamwork at Hollywood went further than the requirements of the job. Officers upheld the blue wall of silence in the face of misconduct investigations, regularly telling each other to "Admit nothing. Deny everything. Demand proof."[20] A diary Lazarus started while working at Hollywood documents her acculturation to the department, showing her increasingly identifying with the LAPD above all, accepted by fellow officers for her efforts to fit in[21] and maintaining a detached, often nonchalant, attitude when responding to sometimes brutal crimes and their traumatized victims.[22]

Lazarus told the friend that her training officer, James Tomer, regularly sexually propositioned her; her repeated refusals led to workplace rumors that she was a lesbian. Later, Tomer faced departmental charges of narcotics theft. Lazarus was called to testify at his disciplinary hearing; Tomer was acquitted and later won a verdict against the department over the charges in federal court, arguing he had been framed, the first time the department had been successfully sued over an internal affairs investigation[23] (the verdict was later overturned on appeal).[24] After a year on probation, she was promoted and transferred to the quieter[b] Devonshire division, covering Northridge, Chatsworth and Granada Hills, in March 1985.[26] She rented the spare bedroom in her condo to a fellow officer.[27]

Ruetten Rasmussen relationship and effect on Lazarus
Lazarus had thrown Ruetten a surprise party on his 25th birthday, unaware that he had been dating other women or that he had developed a serious relationship with Rasmussen.[28] In a May 1985 diary entry, she mentions visiting Ruetten and his girlfriend, the first time she met Rasmussen.[29] The following month Lazarus was depressed after learning that Ruetten and Rasmussen had become engaged a week earlier. In her diary, she wrote, "I really don't feel like working. I found out that John is getting married ... My concentration is like -10".[30] Later that night, she awoke her roommate, a fellow officer, to commiserate.[28] He recalled later that he had never seen her cry until then.[30] Lazarus visited Ruetten at his condo later that month, shortly before he moved in with his fiancée. The two had sex—"to give her closure", Ruetten testified years later[4]—for the last time before Rasmussen's death. He did not tell Rasmussen about the encounter.[31] Lazarus was still not over Ruetten. In August she wrote to her mother: "I'm truly in love with John and the past year has really torn me up ... I wish it didn't end the way it did, and I don't think I'll ever understand his decision."[18] An LAPD women's basketball teammate at the California Police Olympics around that time recalls Lazarus mentioning a boyfriend from UCLA named John.[32]

Sometime in July or August, on a day off, Lazarus visited Rasmussen at work, wearing tight short shorts and a tank top. Lazarus told her that she and John were still having sex and that when their marriage failed, "I'll be there to pick up the pieces". Rasmussen responded that "your services won't be needed." Lazarus left saying "If I can't have John, no one else will. Including you." While Rasmussen did not report the incident to hospital security, she complained to her father about it and confronted Ruetten when he returned from work that evening. He admitted the infidelity and asked Sherri to forgive him, but did not contact Lazarus or take any action to prevent further contact as from his perspective any relationship with her was long over.[33][34][c]

Ruetten and Rasmussen married in November as planned.[18] Her concerns about Lazarus lingered. Since Ruetten had moved in with her they had been receiving occasional hang-up calls.[36][d] Before leaving on their honeymoon near the end of the year, the couple installed a burglar alarm in their condo. Sherri set the alarm code as 1123, their wedding anniversary, which a friend advised her was a poor choice.[38] Ruetten left Dataproducts for Micropolis, a hard drive manufacturer in Chatsworth.[39]

In late 1985, Lazarus seemed to her friends and coworkers to be moving on from Ruetten. She began working part-time security at Los Angeles Pierce College in Woodland Hills and dated again.[40] Near the end of the year she drafted a personal ad in her diary.[41] But in January, Lazarus brought her skis to the condo and asked Ruetten to wax them. Despite Rasmussen's objections, he complied. Rasmussen felt that the visit and request was strange, especially since Lazarus was dressed in flattering workout clothes. After Lazarus left, Rasmussen asked her husband if their relationship was truly over. Ruetten convinced her the two were just friends. After failing to persuade her husband to not wax the skis and just return them, Rasmussen told him not to drop them off at Lazarus's apartment. A few days later, after Ruetten had left for work, Lazarus returned to pick up the waxed skis, in uniform and armed.[42][18]

On another occasion later that month, Rasmussen again confronted Lazarus when, after Ruetten had gone to work one morning, Rasmussen found her in the couple's living room, in uniform. Lazarus left after being firmly asked. Rasmussen told her father about the incident but not Ruetten, and did not believe reporting it to the police would be effective.[43] On a visit with her parents in Tucson in early February, Nels Rasmussen tried to persuade John to move there, assuring his son-in-law he could get him an engineering position through connections.[44] Sherri again confided in her father, after they returned to Los Angeles, telling him she feared that Lazarus was stalking her on the street,[34] sometimes posing as a young man.[45]

On February 14, 1986, Lazarus's roommate moved out. A week later, on February 23, one of Ruetten's friends from UCLA visited the couple's condo. Unlike their usual habit of entering through the garage, the front door was used during the visit, and Ruetten inadvertently left it unlocked afterward.[46]

Crime
On the morning of February 24, 1986, Ruetten left the couple's condominium on Balboa Boulevard[4] at approximately 7:20 a.m. for work.[47] Rasmussen had been scheduled to deliver a motivational presentation at Glendale Adventist Medical Center, a workplace initiative she reportedly viewed as ineffective. She told Ruetten that she might call in sick instead, citing a back injury she sustained during an aerobics class the previous day.[48] After making a few stops, Ruetten arrived at Micropolis about 30 minutes later.[49]

At 9:45, a neighbor noticed that the Ruettens' garage door was open, with no car visible. Approximately 15 minutes later, Ruetten made the first of several unanswered calls home over the course of the day. He found it strange that the answering machine did not respond, since both of them turned it on when leaving the condo unoccupied. Rasmussen's sister, a nurse at another local hospital, also called, getting no answer.[49] At noon, two men, who the neighbor believed were gardeners in the compound, gave her and her husband a purse they had found, which turned out to be Rasmussen's.[50] A maid cleaning a nearby unit said she heard something that sounded like two people fighting, and then something falling, at around 12:30 p.m.[51]

When Ruetten returned home in the evening, he found broken glass on the driveway. Rasmussen's BMW 318i was missing. Ruetten later told investigators it was unusual for her to have left the condo without telling him, given her plans to stay home that morning.[18] Inside, Ruetten found Rasmussen dead on the living room floor, shot three times. There were signs of a struggle, such as a broken porcelain vase, a bloody handprint next to the burglar alarm's panic button, and a toppled credenza. It appeared that someone had attempted to bind Rasmussen.[e] She had defensive wounds and a bruise on her face that appeared to have been inflicted by the muzzle of a gun.[53] The gun had been fired through a quilted blanket, apparently to muffle the sound. The investigating criminalist also observed a bite mark on Rasmussen's arm and took a swab from it for blood typing.[9]

Investigations
Initial investigation
LAPD detectives investigating the case quickly concluded that Rasmussen had been surprised and killed by a burglar. Rasmussen's attire—a bathrobe, nightgown, and underwear—suggested she was not expecting visitors. Although a maid in a neighboring unit reported hearing screaming and fighting, she did not recall hearing gunshots. She thought the whole event had been a domestic dispute and did not call the police. It appeared that the perpetrator had been in the process of taking the electronic equipment left in a stack at the base of the stairs when Rasmussen came upon them. Jewelry had been left behind and the vehicle taken.[9] The only other item that appeared to have been taken from the home was the couple's marriage license,[48] which Sherri's mother believed she might have been carrying in her purse to facilitate consolidating her and Ruetten's bank accounts.[54] An autopsy found all the bullets inflicted injuries severe enough to have resulted in death by themselves shortly afterward. Fifteen separate injuries were noted on Rasmussen's face, including the apparent muzzle bruise and a laceration of the frenulum between the upper lip and jaw, suggesting she was struck forcefully in that area.[55]

Lead detective Lyle Mayer considered other possibilities. He quickly ruled out Ruetten as a suspect.[28] Nels Rasmussen and his wife, Loretta, told Mayer about Lazarus's harassment, and saw that he made a note of it. Ruetten later told police that he and Sherri had never discussed Lazarus.[9][34] Ruetten said later that he told the detectives about Lazarus when they raised the possibility of a jealous former lover at a second interview the morning after the crime.[56] However, he did not tell them about his final sexual encounter with her, nor the confrontation where she told Rasmussen about that liaison.[57][f] Nels Rasmussen also told the police about her, but Mayer dismissed it, telling him "there's nothing there" in Nels's recollection.[58]

The police remained focused on the possibility of burglary, especially in light of one reported later in the same area, in which one of the two reported suspects had been carrying a gun, possibly a .38 like the one that had fired the three bullets,[59] later identified by experts as Federal .38J Plus-P, into Rasmussen's chest.[53] But the crime had some atypical elements for a burglary. Mayer noted that it was unusual for burglars to steal a car after the crime; they usually brought their own vehicle to load stolen goods in. To him, this meant there were likely two burglars.[60] His partner, Steve Hooks, found the bite mark unusual, as bites during struggles are much more commonly inflicted by women, while the majority of burglars are men. However, because men have bitten opponents during fights as well, the burglary theory stood.[18]

Eleven days after Rasmussen's death, her BMW was recovered after an officer saw it parked on the street near the intersection of Zombar Avenue and Cohasset Street, in a residential neighborhood roughly two miles (3.2 km) east of the Ruettens' condo. It was unlocked with the keys in the ignition.[g] While crime-scene technicians found some possible bloodstains and fiber evidence in the car along with four fingerprints that were neither Rasmussen's nor her husband's,[62] a search of the area yielded no other evidence. Whether the car had been there for the entire time since the killing has never been determined. Nonetheless, Mayer believed that location strengthened the burglary theory since burglars were known to live in that area.[63][h] In April, a man arrested after a burglary in progress at a similar condo in Van Nuys where a stereo was taken, appeared to Mayer to be the prime suspect in the case, but was ruled out when his fingerprints did not match.[65]

Cold case
The suspected burglars to whom detectives ascribed the crime remained at large, despite a follow-up newspaper story eight months later and a reward offered by the Rasmussen family. The LAPD, preoccupied with the violence resulting from gang wars and the crack epidemic plaguing the city at the time, was unable to devote much more attention to the case. The Rasmussens said that detectives at the Van Nuys office were often unhelpful when the family called, hanging up or putting them on indefinite hold.[18]

Ruetten briefly returned to the condo, but moved out after a month when it became too much psychologically and financially. After taking a leave of absence from Micropolis, he moved back to live with his parents in San Diego.[66] Lazarus briefly reunited with Ruetten on a 1989 Hawaii vacation; he recounted later having seen her on occasion over the next two years and occasionally having sex.[67] Mayer's notes show that Ruetten had called him and asked if he was absolutely sure there was no evidence linking Lazarus with his late wife's death.[18] In 1991, after moving back to the Los Angeles area for another job, Ruetten met his second wife.[68]

A year after the crime, the frustrated Rasmussen family reiterated their reward offer at a news conference and called for more action. Nels wrote to Daryl Gates, then chief of the LAPD, about the possibility that Lazarus might have been involved.[34] Detectives told him he "watch[ed] too much television."[9] He continued to publicize the reward, and later worked with the short-lived television series Murder One on a segment inspired by the case.[18] In 1988 Sherri's other sister suggested true-crime writer Anne Rule, a fellow Seattle resident, write a book about the case. Rule was interested, and Sherri's parents met with her. Nels told her about the Lazarus angle, and Rule said she would have a friend, a retired LAPD detective who assisted her with research, ask around. A few days later, Rule told Nels that this man had told her that the case was "too hot to handle" so she would not write about it.[69]

Nels in particular was unconvinced that Sherri, who was six feet (1.8 m) tall, had a large frame, and was in good physical shape – had been the victim of a botched burglary. It would have been a struggle for anyone to subdue her in close quarters. Mayer had told him at one point that the events lasted from 45 minutes to an hour and a half, a long time for burglars who were believed to be primarily after items of value in the home.[18] Rasmussen further doubted that any struggle would have lasted that long if the assailant had been male.[70] Further, whoever shot his daughter had fired directly into her chest at close range and taken the trouble to muffle the shot with the quilt. This suggested that the killing was deliberate and not the accidental byproduct of a struggle.[18]

Mayer retired in 1991 and Hooks was reassigned to another division a few years later.[71] The new detective assigned to the case told Nels Rasmussen that he was unable to follow up on Mayer's notes and did not think that any new leads would emerge. Nels was rebuffed again in 1993 when he offered to pay for DNA testing on the evidence, now that the technology was available. He was told that the police had to have a suspect in order to proceed with testing (In October of that year, LAPD records show, detectives showed renewed interest in the case after a year and a half when nothing was done[72]). In 1997 a former coworker of Sherri's secretary contacted police about how, some time after the killing, the woman recounted to her the January 1986 incident where Lazarus had confronted Sherri in her office, but no attempt was made to follow up before the secretary died three years later.[73]

Lazarus continued her LAPD career. She also started her own private investigation firm, Unique Investigations.[48] In 1994, after stints at the department's Drug Abuse Resistance Education (DARE) and background investigations of new recruits,[74] she was promoted to detective.[4] Three years later, she married a fellow officer she met while teaching a DARE class in Oregon and adopted a daughter with him, moving back to Simi Valley; he later joined the LAPD as well. At work she became an instructor at the police academy.[4] For two periods during the 1990s, she was assigned to the Van Nuys division, where the records from the original investigation of Sherri Rasmussen's murder were still on file. Between her tenures at Van Nuys she was assigned to the LAPD's internal affairs division for a year. Department records show that in both the background and internal affairs positions she distinguished herself with the thoroughness and tenacity of her investigations.[75]
"""

### 1. GAME SYSTEM

- GameState
- GameController

Game state

In [8]:
class GameState:

    def __init__(self):

        self.phase = 0
        self.progress = 0.0
        self.pressure = 0.0

        self.question_count = 0

        self.game_over = False

In [9]:
# initialize game state
state = GameState()

Game Controller

In [10]:
class GameController:

    def __init__(self):

        # maximum pressure before suspect shuts down
        self.max_pressure = 1.1

        # progress required to move to next phase
        self.phase_thresholds = {
            0: 1.2,
            1: 2.0,
            2: 2.6,
            3: 3.2,
            4: 3.0
        }

        # question strategy classification
        self.strategy_order = {
            0: "rapport",
            1: "neutral",
            2: "probing",
            3: "confrontational",
            4: "accusatory"
        }

        # limits per interrogation phase
        self.phase_limits = {

            0: {
                "max_strategy": 1,
                "max_aggressiveness": 1,
                "max_pressure": 0.3
            },

            1: {
                "max_strategy": 2,
                "max_aggressiveness": 2,
                "max_pressure": 0.4
            },

            2: {
                "max_strategy": 2,
                "max_aggressiveness": 2,
                "max_pressure": 0.5
            },

            3: {
                "max_strategy": 3,
                "max_aggressiveness": 3,
                "max_pressure": 0.65
            },

            4: {
                "max_strategy": 3,
                "max_aggressiveness": 3,
                "max_pressure": 0.8
            },

            5: {
                "max_strategy": 4,
                "max_aggressiveness": 4,
                "max_pressure": 1.0
            }
        }


    # case briefing shown at the beginning of the game
    def get_case_briefing(self):

        return """
CASE BRIEFING

It's 2009. You are a homicide detective conducting an interrogation.

The woman sitting across from you is Stephanie Lazarus, an LAPD officer.

You are investigating the 1986 murder of Sherri Rasmussen. The case went cold for many years, but new DNA evidence has recently reopened the investigation and Stephanie is the primary suspect.

Sherri Rasmussen was married to John Ruetten. Before their marriage, John had a relationship with Stephanie Lazarus while they were both students at UCLA.

Sherri was found beaten and shot in the condominium she shared with John. At the time, the case was believed to be a burglary gone wrong, and it remained unsolved.

New evidence has now reopened the investigation.

Stephanie has voluntarily agreed to speak with you today. 

Your objective is to question Stephanie about her relationship with John and her connection to Sherri Rasmussen.

You already possess several investigative leads. They range from neutral background information to highly incriminating evidence.

It is important that you explore them carefully and escalate the conversation gradually.

Investigative leads (ordered from least to most confrontational):

1. Relationship with John Ruetten  
   Stephanie dated John while they were students at UCLA.

2. Contact with Sherri Rasmussen  
   There are indications Stephanie may have confronted Sherri at her workplace.

3. Witness testimony  
   Statements suggest tension in public between Stephanie and Sherri.

4. Possible weapon connection  
   The victim was killed with a .38 caliber revolver — similar to the one Stephanie owned.

5. DNA evidence  
   Saliva recovered from a bite mark on the victim that could be linked to Stephanie.

Your strategy should be to let Stephanie talk, ask open questions first, get gradually more specific and allow her to commit to statements that can later be contradicted.

Once enough inconsistencies appear, you may confront her with the strongest evidence.

Be careful. If you push too aggressively too early, Stephanie may become defensive and shut down the interrogation.

"""


    # hints guiding the player during each interrogation phase
    def get_phase_hint(self, phase):

        hints = {

            0: """
INTERROGATION PHASE: INITIAL CONTACT

Start with small talk.

Build rapport and establish a relaxed tone before moving into sensitive topics.
""",

            1: """
INTERROGATION PHASE: BACKGROUND

Explore Stephanie's past with John Ruetten.

Focus on how they met, their time at UCLA, and whether they stayed in contact.
""",

            2: """
INTERROGATION PHASE: CONNECTION TO SHERRI

Shift the conversation toward Sherri Rasmussen.

Find out whether Stephanie knew her or had any interaction with her.
""",

            3: """
INTERROGATION PHASE: TIMELINE

Ask about specific moments and events.

Look for contradictions or suspicious details.
""",

            4: """
INTERROGATION PHASE: CONFRONTATION

You may begin confronting Stephanie with inconsistencies.

Carefully challenge statements that don't match the known facts.
""",

            5: """
INTERROGATION PHASE: ACCUSATION

You now have enough context to confront Stephanie directly.
"""
        }

        return hints.get(phase, "")


    # evaluate whether questioning strategy fits the current phase
    def evaluate_strategy(self, state, strategy, aggressiveness):

        limits = self.phase_limits[state.phase]

        strategy_gap = strategy - limits["max_strategy"]
        aggr_gap = aggressiveness - limits["max_aggressiveness"]

        gap = max(strategy_gap, aggr_gap)

        if gap <= 0:
            return "aligned"

        elif gap == 1:
            return "slightly_advanced"

        else:
            return "too_advanced"


    # update interrogation progress
    def update_progress(self, state, progress_delta):

        state.progress += progress_delta
        state.progress = min(state.progress, 5.0)


    # update pressure depending on questioning strategy
    def update_pressure(self, state, evaluation):

        if evaluation == "aligned":
            state.pressure -= 0.05

        elif evaluation == "slightly_advanced":
            state.pressure += 0.1

        elif evaluation == "too_advanced":
            state.pressure += 0.25

        state.pressure = max(0.0, min(state.pressure, self.max_pressure))


    # handle phase transitions and victory condition
    def check_phase_transition(self, state, player_question):

        if state.phase == 0:

            if state.question_count >= 2:

                state.phase = 1
                state.progress = 0

                return "Phase change: background questions."


        if state.phase == 1:

            if state.progress >= self.phase_thresholds[1]:

                state.phase = 2
                state.progress = 0

                return "Phase change: discussion about Sherri."


        if state.phase == 2:

            if state.progress >= self.phase_thresholds[2]:

                state.phase = 3
                state.progress = 0

                return "Phase change: probing inconsistencies."


        if state.phase == 3:

            if state.progress >= self.phase_thresholds[3]:

                state.phase = 4
                state.progress = 0

                return "Phase change: confronting with evidence."


        if state.phase == 4:

            if state.progress >= self.phase_thresholds[4]:

                state.phase = 5
                state.progress = 0

                return "Phase change: direct accusation."


        # final condition (same ending for both triggers)
        if state.phase == 5:

            if (
                state.progress >= self.phase_thresholds[4]
                or "dna" in player_question.lower()
            ):

                state.game_over = True

                return """
CASE SOLVED

Excellent work, detective.

Through careful questioning and strategic pressure, you managed to expose contradictions in Stephanie Lazarus's story.

You now have enough evidence to move forward with the prosecution.

Watch the real interrogation moment here:
https://youtu.be/WLSNPkf8RCU?t=1305
"""

        return None


    # suspect shuts down interrogation if pressure becomes too high
    def check_game_over(self, state):

        if state.pressure >= self.max_pressure:

            state.game_over = True

            return "Stephanie requests a lawyer. Interrogation ends. GAME OVER"

        return None

In [11]:
# initialize controller
controller = GameController()

### 2. AGENTIC SYSTEM


#### Design


```text
Player Question
      │
      ▼
[Inject Game State]
(state.phase, state.pressure)

      │
      ▼
[Input Preparation]
(question | phase | pressure | chat_history)

      │
      ▼
┌──────────────────────────── PARALLEL ANALYSIS ────────────────────────────┐
│                                                                           │
│   Question Evaluation      Suspect Pattern Retrieval      Strategy Classifier
│   ──────────────────       ─────────────────────────       ──────────────────
│   semantic similarity      retrieve interrogation         detect questioning
│   phase alignment          dialogue patterns              strategy
│   progress scoring         behavioural context            (rapport / pressure)
│                                                                           │
└───────────────────────────────────────────────────────────────────────────┘

      │
      ▼
[Suspect Response Generator]
LLM(roleplay) ← question + chunks + evaluation + strategy + history

      │
      ▼
[Game State Update]
update interrogation mechanics:
phase progression | pressure | interrogation progress

      │
      ▼
[Game Over Router] ──► check_game_over(state)
        │
        ├── TRUE  → forced shutdown response + game over
        │
        └── FALSE → continue interrogation

      │
      ▼
[Detective Instinct System]
generate player feedback

      │
      ▼
Final Output
response | feedback | state_update | game_over
```

###### Design Principles

* **LLM handles dialogue only**
* **Tools handle reasoning and game mechanics**
* **Vector retrieval acts as interrogation knowledge**
* **Game controller enforces narrative pacing**

This creates a **controlled agentic interrogation system** where semantic similarity drives investigation progress while maintaining structured gameplay.


#### Tools

- evaluate_question_tool
- retrieve_suspect_patterns_tool
- classify_question_strategy_tool
- generate_suspect_response_tool
- update_game_state_tool
- detective_feedback_tool

##### Evaluate question tool

player_question > detective_vector_store > evaluate phase alignment > generate outputs

In [12]:
@tool
def evaluate_question_tool(input_data: dict) -> dict:
    """
    Evaluate the detective's question using the detective vector database.

    Produces signals used for:
    - interrogation progress
    - suspect behaviour
    - player feedback
    """

    question = input_data["question"]

    # retrieve similar interrogation questions (hide final parts of the interrogation)
    allowed_phases = list(range(0, state.phase + 3))
    allowed_phases = [p for p in allowed_phases if p >= 0]
    # retrieval
    results = detective_vector_store.similarity_search_with_score(
        question,
        k=5,
        filter={"phase": {"$in": allowed_phases}}
    ) 
    retrieved_phases = []
    similarities = []
    
    

    # get similarity scores
    for doc, score in results:
        phase = doc.metadata.get("phase", None)
        if phase is not None:
            retrieved_phases.append(int(phase))
        similarity = score 
        similarities.append(similarity)
  

    # determine which phases are acceptable for conversational alignment
    # alignment accepts all previous phases plus the next phase
    valid_alignment_phases = list(range(0, state.phase + 2))
    aligned_similarities = [
        s for s, p in zip(similarities, retrieved_phases)
        if p in valid_alignment_phases
    ]
    if len(retrieved_phases) == 0:
        phase_alignment_ratio = 0
        mean_similarity = 0

    else:
        phase_alignment_ratio = len(aligned_similarities) / len(retrieved_phases)
        if aligned_similarities:
            mean_similarity = sum(aligned_similarities) / len(aligned_similarities)
        else:
            mean_similarity = 0


    # define phases allowed to generate interrogation progress
    # allows slight backward or forward exploration
    progress_phases = [
        state.phase - 1,
        state.phase,
        state.phase + 1
    ]

    # for safety (and phase 0) eliminates negative phases.
    progress_phases = [p for p in progress_phases if p >= 0]


    # apply phase weighting to favour questions aligned with the current phase
    weighted_scores = []
    for similarity, phase in zip(similarities, retrieved_phases):
        if phase not in progress_phases:
            continue
        if phase == state.phase:
            weight = 1.0
        elif phase == state.phase + 1:
            weight = 0.7
        elif phase == state.phase - 1:
            weight = 0.6
        else:
            weight = 0
        weighted_scores.append(similarity * weight)


    if weighted_scores:
        max_similarity = max(weighted_scores)
    else:
        max_similarity = 0

    # convert similarity into interrogation progress
    SIMILARITY_THRESHOLD = 0.3
    progress_delta = max_similarity if max_similarity >= SIMILARITY_THRESHOLD else 0


    return {
        "phase_alignment_ratio": phase_alignment_ratio,
        "mean_similarity": mean_similarity,
        "max_similarity": max_similarity,
        "progress_delta": progress_delta,
        "retrieved_phases": retrieved_phases
    }

##### Retrieve suspect patterns tool

player_question > suspect_vector_store > retrieve interrogation patterns

In [13]:
@tool
def retrieve_suspect_patterns_tool(input_data: dict) -> list:
    """
    Retrieve suspect dialogue patterns related to the player's question.
    """
    question = input_data["question"]
    phase = input_data["phase"]

    k = 3

    results = suspect_vector_store.similarity_search(
        question,
        k=k,
    )

    chunks = [doc.page_content for doc in results]

    return chunks

##### Classify question strategy tool

Classify detective intent and tone with LLM

In [14]:
class StrategyOutput(BaseModel):
    strategy: int
    aggressiveness: int

@tool
def classify_question_strategy_tool(input_data: dict) -> dict:
    """
    Classify the interrogation strategy and aggressiveness of the detective's question.
    """

    question = input_data["question"]

    prompt = f"""
You are analyzing a police interrogation question.

Classify the strategy and aggressiveness of the question.

Strategies scale:
0 = rapport, greeting or building comfort
1 = neutral, general conversation
2 = seeking information
3 = confrontational, challenging inconsistencies
4 = accusatory, directly accusing the suspect

Aggressiveness scale:
0 = friendly talk
1 = polite or neutral language
2 = firm talk
3 = accusatory, aggressive or threatening language
4 = hostile, rude, cuzz words, violent or phisycally threatening language

Return two integers:
strategy
aggressiveness

Question:
{question}
"""

    structured_llm = llm.with_structured_output(StrategyOutput)

    result = structured_llm.invoke(prompt)

    return {
        "strategy": result.strategy,
        "aggressiveness": result.aggressiveness
    }

##### Generate suspect response tool

In [15]:
@tool
def generate_suspect_response_tool(input_data: dict) -> str:
    """
    Generate Stephanie Lazarus's response using case facts,
    interrogation patterns and conversation history.
    """

    # player question
    question = input_data["question"]

    # retrieved interrogation behaviour patterns
    chunks = input_data["chunks"]

    # previous conversation
    chat_history = input_data["chat_history"]

    # evaluation signals from question evaluation
    evaluation = input_data["evaluation"]

    # interrogation strategy classification
    strategy_data = input_data["strategy"]
    strategy = strategy_data["strategy"]
    aggressiveness = strategy_data["aggressiveness"]

    # convert chat history into prompt text
    history_text = ""
    for msg in chat_history:
        history_text += f"{msg.content}\n"

    patterns_text = "\n\n".join(chunks)

    # alignment ratio indicates how relevant the question is to the current interrogation phase
    alignment = evaluation["phase_alignment_ratio"]

    # phase limits for strategy and aggressiveness
    limits = controller.phase_limits[state.phase]

    too_advanced = (
        strategy > limits["max_strategy"]
        or aggressiveness > limits["max_aggressiveness"]
    )


    # BEHAVIOUR BASED ON QUESTION
    behaviour_signal = ""

    # aggressive or premature interrogation strategy
    if too_advanced:
        behaviour_signal = """
The detective used an aggressive or premature interrogation strategy.
You become suspicious and defensive.
You question the detective's intentions and avoid giving useful information.
"""

    # question unrelated to the current conversation
    elif alignment < 0.2:
        behaviour_signal = """
The detective's question feels unrelated or confusing.
Respond vaguely and avoid giving useful information.
"""

    # somewhat weak question
    elif alignment < 0.3:
        behaviour_signal = """
The question is somewhat unfocused.
Respond cautiously and provide minimal details.
"""

    # normal interrogation question
    elif alignment < 0.6:
        behaviour_signal = """
The question is reasonable but not very revealing.
Answer normally while avoiding incriminating information.
"""

    # strong interrogation question
    else:
        behaviour_signal = """
The detective asked a relevant and well-timed question.
Respond naturally and provide plausible details while still avoiding incriminating yourself.
"""


    # Style change over phase
    phase = state.phase

    phase_state = ""

    if phase == 0:

        phase_state = """
    You believe this is a routine conversation.

    You are relaxed, friendly and cooperative.

    Your answers are short and casual.
    """

    elif phase == 1:

        phase_state = """
        You are still relaxed but slightly cautious.

    Your answers remain short and conversational.
    """

    elif phase == 2:

        phase_state = """
    You begin to feel uncomfortable.
    You speak more carefully and give sometimes longer explanations
    to justify or clarify your answers.
    """

    elif phase == 3:

        phase_state = """
    The detective is asking detailed questions about events.
    You start to suspect he knows you're connected to the investigation.
    You become cautious and defensive.
    You can question the purpose of the conversation or ask for it.
    Your answers are slightly longer as you try to explain yourself.
    """

    elif phase >= 4:

        phase_state = """
    The detective is confronting you with inconsistencies.
    You feel under pressure.
    You become defensive and emotionally tense.
    You may justify yourself, challenge the detective, or question the accusations.
    """


    prompt = f"""
You are Stephanie Lazarus, an LAPD detective speaking with another detective.

At the beginning of the conversation you are relaxed and cooperative.
You assume this is a routine conversation.

You do NOT assume you are a suspect.
Stephanie does NOT know what the detective already knows.
Do not introduce topics that the detective has not mentioned.

Keep responses short.
Most responses should be 1–2 sentences. 
Phase 0–1: answers should usually be 1-2 sentences.
Phase 2–3: answers can be 1–3 sentences as Stephanie explains herself.
Phase 4–5: answers may be longer when Stephanie becomes defensive.
In any phase, give longer answers if the detective asks for details.

The case facts are only your memory of what really happened, but you may:
- answer casually
- be slightly cautious
- say you don't remember

Only become defensive if the detective starts suggesting your involvement in the crime.

From phase 3 onwards, 
if the detective exposes contradictions on your previous answers or confronts you with facts that you know are true, 
don't deny them fully, look for possible explanations or justifications without incriminating yourself.

Never confess.
Avoid incriminating yourself.

CASE FACTS
{CASE_FACTS}

CURRENT PHASE
{state.phase}

PRESSURE LEVEL
{state.pressure}

QUESTION QUALITY SIGNAL
{behaviour_signal}

INTERROGATION STATE
{phase_state}

REAL INTERROGATION PATTERNS
{patterns_text}

CONVERSATION HISTORY
{history_text}

Detective: {question}

Stephanie:
"""

    response = llm.invoke(
        prompt,
        stop=["Detective:"]
    )

    return response.content

##### Detective feedback tool

In [ ]:
last_instinct_feedback = None

@tool
def detective_feedback_tool(input_data: dict) -> str:
    """
    Provide feedback to the player about their interrogation strategy.
    """

    global last_instinct_feedback

    evaluation = input_data["evaluation"]
    strategy_data = input_data["strategy"]
    pressure = input_data["pressure"]

    strategy = strategy_data["strategy"]
    aggressiveness = strategy_data["aggressiveness"]

    # alignment = evaluation["phase_alignment_ratio"]
    similarity = evaluation["max_similarity"]

    limits = controller.phase_limits[state.phase]

    too_advanced = (
        strategy > limits["max_strategy"]
        or aggressiveness > limits["max_aggressiveness"]
    )

    feedback = []

    # danger state (highest priority)
    if pressure > 0.8:
        feedback.append(
            "Careful. Stephanie looks close to shutting down."
        )

    # interrogation pacing
    elif too_advanced:
        feedback.append(
            "You were too direct too quickly, putting Stephanie on guard."
        )

    # weak direction
    #elif alignment < 0.3:
    elif similarity < 0.3:
        feedback.append(
            "This line of questioning doesn't seem very useful."
        )

    # strong signal
    elif similarity > 0.65:
        feedback.append(
            "Good instinct. You're on the right track."
        )

    # no signal -> no feedback
    if not feedback:
        return ""

    return " ".join(feedback)

##### Update game state tool

In [17]:
from langchain.tools import tool

@tool
def update_game_state_tool(input_data: dict) -> dict:
    """
    Update the interrogation game state after the suspect responds.
    """

    # variables
    evaluation = input_data["evaluation"]
    question = input_data["question"]

    progress_delta = evaluation["progress_delta"]

    strategy = input_data["strategy"]["strategy"]
    aggressiveness = input_data["strategy"]["aggressiveness"]


    
    # evaluate strategy vs phase rules
    strategy_evaluation = controller.evaluate_strategy(
        state,
        strategy,
        aggressiveness
    )

    # pressure update
    controller.update_pressure(
        state,
        strategy_evaluation
    )


    # progress update (only if aligned)
    if strategy_evaluation == "aligned":
        controller.update_progress(state, progress_delta)


    # question counter
    state.question_count += 1


    # phase change
    phase_message = controller.check_phase_transition(
        state,
        question
    )


    # game over
    game_over_message = controller.check_game_over(state)


    return {
        "phase_message": phase_message,
        "game_over_message": game_over_message,
        "phase": state.phase,
        "progress": state.progress,
        "pressure": state.pressure,
        "strategy_evaluation": strategy_evaluation
    }

#### Pipeline

In [18]:
inject_state = RunnableLambda(
    lambda x: {
        **x,
        "phase": state.phase,
        "pressure": state.pressure
    }
)

pipeline = (

    # Inject game state
    inject_state

    # Prepare inputs
    | RunnablePassthrough.assign(
        question=lambda x: x["question"],
        phase=lambda x: x["phase"],
        pressure=lambda x: x["pressure"],
        chat_history=lambda x: x["chat_history"]
    )

    # Parallel analysis: evaluation + retrieval
    | RunnableParallel(
        question=lambda x: x["question"],
        phase=lambda x: x["phase"],
        pressure=lambda x: x["pressure"],
        chat_history=lambda x: x["chat_history"],

        evaluation=lambda x: evaluate_question_tool.invoke({
            "input_data": {
                "question": x["question"]
            }
        }),

        chunks=lambda x: retrieve_suspect_patterns_tool.invoke({
            "input_data": {
                "question": x["question"],
                "phase": x["phase"]
            }
        }),

        strategy=lambda x: classify_question_strategy_tool.invoke({
            "input_data": {
                "question": x["question"]
            }
        })
    )

    # Generate suspect response
    | RunnablePassthrough.assign(
        response=lambda x: generate_suspect_response_tool.invoke({
            "input_data": {
                "question": x["question"],
                "chunks": x["chunks"],
                "chat_history": x["chat_history"],
                "evaluation": x["evaluation"],
                "strategy": x["strategy"]
            }
        })
    )

    # Update game state
    | RunnablePassthrough.assign(
        state_update=lambda x: update_game_state_tool.invoke({
            "input_data": {
                "evaluation": x["evaluation"],
                "question": x["question"],
                "strategy": x["strategy"]
            }
        })
    )

    # Game over router
    | RunnableLambda(
        lambda x: (
            {
                **x,
                "response": "You know what? I'm done talking. I want a lawyer.",
                "game_over_message": "You pushed too hard. The suspect shuts down. Game Over",
                "game_over": True
            }
            if controller.check_game_over(state)
            else {
                **x,
                "game_over": False
            }
        )
    )

    # Detective Instinct / Feedback
    | RunnablePassthrough.assign(
        feedback=lambda x: (
            None
            if x["game_over"]
            else detective_feedback_tool.invoke({
                "input_data": {
                    "evaluation": x["evaluation"],
                    "pressure": x["state_update"]["pressure"],
                    "strategy": x["strategy"]
                }
            })
        )
    )

    # output
    | RunnableLambda(
    lambda x: {
        **x,
        "output": x["response"]
    }
    )

)

#### Memory

In [19]:
store: dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str):

    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]


agent_with_memory = RunnableWithMessageHistory(
    pipeline,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

### 3. GAME LOOP / START INTERROGATION

In [20]:
session_id = "interrogation_session"

print(controller.get_case_briefing())
print(controller.get_phase_hint(state.phase))

while not state.game_over:
    
    # get question
    player_question = input("\nDetective: ")
    print("\nDetective:", player_question)
    
    # create answer
    result = agent_with_memory.invoke(
        {"question": player_question},
        config={"configurable": {"session_id": session_id}}
    )
    print("\nStephanie:", result["response"])

    # detective feedback
    if result.get("feedback"):
        print("\n[Instinct]:", result["feedback"])

    # game over check
    if result.get("game_over"):
        print("\n", result.get("game_over_message"))
        break
    
    # phase change text
    phase_message = result.get("state_update", {}).get("phase_message")
    if phase_message:
        print("\n", phase_message)
        print(controller.get_phase_hint(state.phase))




CASE BRIEFING

It's 2009. You are a homicide detective conducting an interrogation.

The woman sitting across from you is Stephanie Lazarus, an LAPD officer.

You are investigating the 1986 murder of Sherri Rasmussen. The case went cold for many years, but new DNA evidence has recently reopened the investigation and Stephanie is the primary suspect.

Sherri Rasmussen was married to John Ruetten. Before their marriage, John had a relationship with Stephanie Lazarus while they were both students at UCLA.

Sherri was found beaten and shot in the condominium she shared with John. At the time, the case was believed to be a burglary gone wrong, and it remained unsolved.

New evidence has now reopened the investigation.

Stephanie has voluntarily agreed to speak with you today. 

Your objective is to question Stephanie about her relationship with John and her connection to Sherri Rasmussen.

You already possess several investigative leads. They range from neutral background information to hi